# 01 · Evaluation — Export PyTorch Checkpoint → ONNX

Load the best `.pth` checkpoint, rebuild the `DeepfakeClassifier` architecture, and export to ONNX with a fixed input shape (`1 × 3 × 224 × 224`). Verify the exported graph produces identical logits to the PyTorch model.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import onnxruntime as ort
import numpy as np
import onnx
import json

In [2]:
class DeepfakeClassifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pretrained else None
        )

        for p in resnet.parameters():
            p.requires_grad = False
        # for p in resnet.layer2.parameters():
        #     p.requires_grad = True
        for p in resnet.layer3.parameters():
            p.requires_grad = True
        for p in resnet.layer4.parameters():
            p.requires_grad = True

        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.avgpool = resnet.avgpool

        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(2048, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.head(x).squeeze(1)

### Cargar Modelo .PTH

In [9]:
PTH_PATH  = './05 Checkpoints/best_model_r50.pth'
ONNX_PATH = './09 Final Model/faceswapp_detector.onnx'




model = DeepfakeClassifier(pretrained=False)
model.load_state_dict(torch.load(PTH_PATH, map_location="cpu",weights_only=False))

model.eval();

### Exportar ONNX

In [11]:
dummy_input = torch.randn(1, 3, 224, 224)  # batch=1, RGB, 224x224

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    opset_version=17,
    input_names=["image"],
    output_names=["logit"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "logit": {0: "batch_size"},
    },
    do_constant_folding=True,
)
print(f"ONNX exportado en: {ONNX_PATH}")

/tmp/ipykernel_23043/957289734.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0528 19:47:18.876000 23043 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0528 19:47:19.583000 23043 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' 

[torch.onnx] Obtain model graph for `DeepfakeClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DeepfakeClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/alloc/data/.cache/virtualenvs/app-C8FdE8xm-py3.10/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/alloc/data/.cache/virtualenvs/app-C8FdE8xm-py3.10/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/alloc/data/.cache/virtualenvs/app-C8FdE8xm-py3.10/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
  File "/alloc/data/.cache/virtualenvs/app-C8FdE8xm-py3.10/lib/python3.10/site-packages/onnx/version_converter.py", line 39, in convert_version
    converted_m

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX exportado en: ./09 Final Model/faceswapp_detector.onnx


In [12]:
# torch.onnx.export genera 2 archivos onnx y onnx.data
# cargo y los uno
"""
pytorch > 2.1, este seria el codigo
dummy_input = torch.randn(1, 3, 224, 224)

export_output = torch.onnx.dynamo_export(backbone, dummy_input)
export_output.save(ONNX_PATH)

print(f"ONNX exportado en: {ONNX_PATH}")
"""
model_onnx = onnx.load(ONNX_PATH)
onnx.save(model_onnx, ONNX_PATH, save_as_external_data=False)

### Inferencia de prueba

In [13]:
sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
dummy_np = np.random.randn(1, 3, 224, 224).astype(np.float32)
output = sess.run(["logit"], {"image": dummy_np})[0]

print(f"Shape output : {output.shape}")          # (1, 512)

# Comparar salida PyTorch vs ONNX 
with torch.no_grad():
  torch_out = model(torch.tensor(dummy_np)).numpy()

max_diff = np.abs(torch_out - output).max()
print(f"Diferencia máx PyTorch vs ONNX: {max_diff:.2e}")  # debe ser < 1e-5

Shape output : (1,)
Diferencia máx PyTorch vs ONNX: 2.38e-07
